# AeroClean — YOLO26n Training Notebook

**Purpose:** Train a YOLO26n model to detect `clean_board` and `dirty_board` classes,  
then export to NCNN format for deployment on a Raspberry Pi 5.

**Runtime:** Set runtime to **GPU → T4** before running  
*(Runtime → Change runtime type → T4 GPU)*

---

## Workflow overview
1. Mount Google Drive (stores your dataset and checkpoints)
2. Install dependencies
3. Verify GPU
4. Prepare dataset — **Option A:** Label Studio export  **or**  **Option B:** Roboflow
5. Train YOLO26n
6. Evaluate — view mAP, loss curves, confusion matrix
7. Export to NCNN
8. Download weights for deployment on the Pi

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# We'll store everything under this folder in your Drive:
DRIVE_ROOT = '/content/drive/MyDrive/AeroClean'

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

## Step 2 — Install dependencies

In [ ]:
!pip install ultralytics --quiet
import ultralytics
ultralytics.checks()   # prints version + GPU info

## Step 3 — Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU found. Training will be slow. Change runtime to GPU.')

## Step 4 — Upload your dataset

### Option A — Label Studio *(default)*
1. In Label Studio open your project → **Export** → choose **YOLO** format → download the zip
2. Run the **Option A cell** below, click **Choose Files**, and select that zip
3. The cell splits images automatically into train / val / test and writes `dataset.yaml`

> Label Studio's YOLO export contains flat `images/` and `labels/` folders with a `classes.txt`.  
> The cell handles the 70 / 20 / 10 split for you — no manual prep needed.

### Option B — Roboflow
If you labeled in Roboflow instead, paste your export snippet in the **Option B cell** below.  
Roboflow already provides the train / val / test split and `data.yaml`.

The dataset must end up in YOLO format:
```
dataset/
  images/
    train/  val/  test/
  labels/
    train/  val/  test/
  dataset.yaml
```

In [ ]:
# ── Option A: Label Studio YOLO export ────────────────────────────────────
# Export from Label Studio: your project → Export → YOLO → download zip.
# Upload that zip here. The cell splits images into train/val/test and
# writes dataset.yaml — skip Step 5 after running this cell.

import os, zipfile, shutil, random, yaml
from google.colab import files

print('Select your Label Studio YOLO export zip...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

# Extract to a temp dir
TMP_DIR = '/content/ls_raw'
shutil.rmtree(TMP_DIR, ignore_errors=True)
os.makedirs(TMP_DIR)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(TMP_DIR)

# Locate images and labels
img_src = os.path.join(TMP_DIR, 'images')
lbl_src = os.path.join(TMP_DIR, 'labels')

# Class names from classes.txt, fall back to AeroClean defaults
classes_file = os.path.join(TMP_DIR, 'classes.txt')
if os.path.exists(classes_file):
    with open(classes_file) as f:
        class_names = [l.strip() for l in f if l.strip()]
else:
    class_names = ['clean_board', 'dirty_board']
print(f'Classes: {class_names}')

# 70 / 20 / 10 train / val / test split
images = sorted([f for f in os.listdir(img_src) if not f.startswith('.')])
random.seed(42)
random.shuffle(images)
n       = len(images)
n_train = int(0.7 * n)
n_val   = int(0.2 * n)
splits  = {
    'train': images[:n_train],
    'val':   images[n_train:n_train + n_val],
    'test':  images[n_train + n_val:],
}
print(f'Split — train: {len(splits["train"])}  val: {len(splits["val"])}  test: {len(splits["test"])}')

DATASET_DIR = os.path.join(DRIVE_ROOT, 'dataset')
shutil.rmtree(DATASET_DIR, ignore_errors=True)

for split, imgs in splits.items():
    os.makedirs(os.path.join(DATASET_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(DATASET_DIR, 'labels', split), exist_ok=True)
    for img_file in imgs:
        shutil.copy(os.path.join(img_src, img_file),
                    os.path.join(DATASET_DIR, 'images', split, img_file))
        lbl_file = os.path.splitext(img_file)[0] + '.txt'
        lbl_path = os.path.join(lbl_src, lbl_file)
        if os.path.exists(lbl_path):
            shutil.copy(lbl_path,
                        os.path.join(DATASET_DIR, 'labels', split, lbl_file))

# Write dataset.yaml
YAML_PATH = os.path.join(DATASET_DIR, 'dataset.yaml')
yaml_data = {
    'path':  DATASET_DIR,
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    len(class_names),
    'names': class_names,
}
with open(YAML_PATH, 'w') as f:
    yaml.dump(yaml_data, f)

print(f'\nDataset ready: {DATASET_DIR}')
print(f'dataset.yaml:  {YAML_PATH}')
print('Skip Step 5 — dataset.yaml already written.')
!ls {DATASET_DIR}

In [ ]:
# ── Option B: Roboflow export (skip if you used Option A) ─────────────────
# !pip install roboflow --quiet
# from roboflow import Roboflow
# rf = Roboflow(api_key='YOUR_API_KEY')
# project = rf.workspace('YOUR_WORKSPACE').project('aeroclean')
# dataset = project.version(1).download('yolov8')   # yolov8 format works for YOLO11
# DATASET_DIR = dataset.location
# print(f'Dataset at {DATASET_DIR}')

## Step 5 — Set dataset.yaml path

> **Skip this step if you used Option A (Label Studio)** — `YAML_PATH` and `dataset.yaml` were already set by that cell.

Run this step only if you used **Option B (Roboflow)**:  
it locates `data.yaml` inside the Roboflow export and updates the `path` key to point to `DATASET_DIR`.

In [ ]:
import yaml

# Path to dataset.yaml (adjust if your zip has a different structure)
YAML_PATH = os.path.join(DATASET_DIR, 'train', 'dataset.yaml')

# Update the 'path' key inside dataset.yaml to point to DATASET_DIR
with open(YAML_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['path'] = DATASET_DIR

with open(YAML_PATH, 'w') as f:
    yaml.dump(cfg, f)

print('dataset.yaml updated:')
print(yaml.dump(cfg))

## Step 6 — Train YOLO26n

**Key hyperparameters:**

| Parameter | Value | Notes |
|---|---|---|
| `epochs` | 100 | Increase to 150–200 if mAP plateaus |
| `imgsz` | 640 | Standard for YOLO26n |
| `batch` | 16 | T4 GPU fits 16 comfortably |
| `patience` | 20 | Early stop if no improvement for 20 epochs |

**Augmentation — conservative profile** (monochrome camera, head-on board, consistent lighting):

| Parameter | Value | Rationale |
|---|---|---|
| `degrees` | 5 | Slight rotation only — board won't be heavily tilted |
| `translate` | 0.05 | Minimal shift — board is always roughly centred in frame |
| `scale` | 0.3 | Mild zoom variance (default 0.5 is too aggressive) |
| `fliplr` | 0.5 | Horizontal flip OK — board looks the same mirrored |
| `flipud` | 0.0 | Off — upside-down board is not a real scenario |
| `shear` | 0.0 | Off — head-on view, no shear expected |
| `perspective` | 0.0 | Off — camera faces board straight on |
| `mosaic` | 1.0 | Keep — helps generalise with small datasets |
| `mixup` | 0.0 | Off — blending classes confuses detector |
| `copy_paste` | 0.0 | Off |
| `hsv_h` | 0.0 | Off — monochrome camera has no hue |
| `hsv_s` | 0.0 | Off — monochrome camera has no saturation |
| `hsv_v` | 0.3 | Mild brightness shift — applies to grayscale luminance |
| `erasing` | 0.1 | Light random erasing — simulates partial occlusion |

Training saves checkpoints to `DRIVE_ROOT/runs/` automatically.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')   # downloads pretrained weights (~2.6MB)

results = model.train(
    data=YAML_PATH,          # path to dataset.yaml — tells YOLO where images/labels are
    epochs=100,              # full passes over the dataset; more = better up to a point
    imgsz=640,               # resize all images to 640×640 before training (YOLO26n standard)
    batch=16,                # images processed per gradient step — T4 GPU handles 16 comfortably
    patience=20,             # stop early if mAP doesn't improve for 20 consecutive epochs
    device=0,                # use GPU 0 (the T4 Colab gives us)
    project=os.path.join(DRIVE_ROOT, 'runs'),  # save checkpoints to Google Drive so they survive session resets
    name='aeroclean_yolo26n',  # subfolder name inside project/ for this run's outputs
    exist_ok=True,           # don't crash if that folder already exists (safe to re-run)

    # ── Augmentation — conservative (monochrome camera, head-on board, consistent lighting) ──
    degrees=5,          # slight rotation only
    translate=0.05,     # minimal shift
    scale=0.3,          # mild zoom variance
    fliplr=0.5,         # horizontal flip OK
    flipud=0.0,         # off — upside-down board is unrealistic
    shear=0.0,          # off — head-on view
    perspective=0.0,    # off — camera faces board straight on
    mosaic=1.0,         # keep — helps with small datasets
    mixup=0.0,          # off — blending classes confuses detector
    copy_paste=0.0,     # off
    hsv_h=0.0,          # off — monochrome camera has no hue
    hsv_s=0.0,          # off — monochrome camera has no saturation
    hsv_v=0.3,          # mild brightness shift — still applies to grayscale
    erasing=0.1,        # light random erasing
)

print('Training complete!')
print('Best weights:', results.save_dir)

## Step 7 — Evaluate the model

In [ ]:
from IPython.display import Image, display
import glob

run_dir = os.path.join(DRIVE_ROOT, 'runs', 'aeroclean_yolo11n')

# Validation metrics (mAP50, mAP50-95, precision, recall)
val_results = model.val(data=YAML_PATH, split='val')
print(f'mAP50:    {val_results.box.map50:.4f}')
print(f'mAP50-95: {val_results.box.map:.4f}')

# Training curves
for img in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    path = os.path.join(run_dir, img)
    if os.path.exists(path):
        print(f'\n--- {img} ---')
        display(Image(path))

## Step 8 — Export to NCNN

NCNN is the fastest inference format on the Raspberry Pi 5 ARM CPU.  
The export produces a folder containing `*.param` and `*.bin` files.

In [ ]:
best_weights = os.path.join(run_dir, 'weights', 'best.pt')
export_model = YOLO(best_weights)

export_path = export_model.export(format='ncnn')
print(f'NCNN model saved to: {export_path}')

## Step 9 — Download weights to your computer

After downloading, copy the `best_ncnn_model/` folder to  
the Pi at `AeroClean/weights/best_ncnn_model/`.

In [ ]:
import shutil
from google.colab import files

ncnn_dir = export_path   # path returned by model.export()
zip_out = '/content/best_ncnn_model.zip'
shutil.make_archive('/content/best_ncnn_model', 'zip', ncnn_dir)

print('Downloading best_ncnn_model.zip ...')
files.download(zip_out)

## Step 10 — Deploy to the Raspberry Pi

```bash
# On your laptop — copy the weights to the Pi:
scp best_ncnn_model.zip pi@<PI_IP>:~/AeroClean/weights/

# On the Pi — unzip:
cd ~/AeroClean/weights
unzip best_ncnn_model.zip -d best_ncnn_model

# Run inference:
cd ~/AeroClean
python main.py --model yolo
```

The `config.json` key `yolo_weights` already points to `weights/best_ncnn_model` by default.